In [1]:
import numpy as np
import pandas as pd


In [ ]:
# Required dataset loading (compact)
import torch
import numpy as np
import pandas as pd

# -------------------------
# Device configuration
# -------------------------
# Options: 'auto', 'cpu', 'cuda'
DEVICE_MODE = 'auto'

if DEVICE_MODE == 'cpu':
    device = torch.device('cpu')
elif DEVICE_MODE == 'cuda':
    if not torch.cuda.is_available():
        raise RuntimeError("DEVICE_MODE='cuda' but CUDA is not available.")
    device = torch.device('cuda')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def _downcast_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Improvement 4: Downcast numeric columns to float32/int32 to halve CPU RAM usage."""
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    return df


# NSL-KDD fresh load (keeps original attack labels for grouping)
nsl_columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment',
    'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted',
    'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'attack', 'level'
 ]

nsl_train = _downcast_dataframe(pd.read_csv('./dataset/nslkdd/KDDTrain+.txt', header=None))
nsl_test = _downcast_dataframe(pd.read_csv('./dataset/nslkdd/KDDTest+.txt', header=None))
nsl_train.columns = nsl_columns
nsl_test.columns = nsl_columns
nsl_kdd_fresh = pd.concat([nsl_train, nsl_test], ignore_index=True)

# CICIDS-2017 and UAVIDS
cicd2017 = _downcast_dataframe(pd.read_csv('./dataset/cicids/Wednesday-workingHours.pcap_ISCX.csv'))
uavids = _downcast_dataframe(pd.read_csv('./dataset/uavids/UAVIDS-2025.csv'))

# Shared dataset registry used by HPO + experiments
datasets = {
    'CICIDS2017': cicd2017,
    'NSL-KDD': nsl_kdd_fresh,
    'UAVIDS': uavids,
}

target_columns = {
    'NSL-KDD': 'attack',
    'CICIDS2017': ' Label',
    'UAVIDS': 'label',
}

print('Loaded datasets:', {k: v.shape for k, v in datasets.items()})
print(f"Selected device mode: {DEVICE_MODE}")
print(f'Using device: {device}')


# Utils and Pipeline Functions

In [ ]:
# Tasks 1, 2, 10: Preprocessing utilities
import random
import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


def set_seed(seed: int = 42) -> None:
    """Task 10: Set random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def preprocess_data(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Task 1: Clean inf/NaN values while keeping all columns (including target).

    Rows are dropped based on inf/NaN in numeric columns only; the original
    index is preserved so callers can reindex target labels safely.
    """
    df = dataframe.copy()
    numeric_cols = df.select_dtypes(include=['number']).columns
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=numeric_cols)
    return df


def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    """Task 2: Build a ColumnTransformer with StandardScaler for numeric columns
    and OneHotEncoder for categorical columns.

    Fit on training data and use transform() on val/test to avoid leakage.
    """
    numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=['number']).columns.tolist()

    transformers = [('num', StandardScaler(), numeric_cols)]
    if categorical_cols:
        transformers.append(
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
        )

    return ColumnTransformer(transformers=transformers, remainder='drop')


In [ ]:
# Task 9: Install PyG dependencies (run once; pinned to torch 2.2 CPU wheels)
# Uncomment the lines below if torch_geometric is not yet installed.
# import sys
# !{sys.executable} -m pip install torch_scatter torch_sparse torch_cluster torch_spline_conv \
#     -f https://data.pyg.org/whl/torch-2.2.0+cpu.html
# !{sys.executable} -m pip install torch-geometric


In [ ]:
# Task 9: Sanity-check imports and version pins
import importlib, sys
import torch

print(f'Python  : {sys.version}')
print(f'PyTorch : {torch.__version__}')

for pkg in ['torch_geometric', 'torch_scatter', 'torch_sparse', 'torch_cluster']:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'installed')
        print(f'{pkg:<20}: {ver}')
    except ImportError:
        print(f'{pkg:<20}: NOT FOUND – run the install cell above')


In [4]:


import torch
from torch_geometric.data import Data

c:\Users\ashis\.conda\envs\tf9\lib\site-packages\torch_geometric\typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
c:\Users\ashis\.conda\envs\tf9\lib\site-packages\torch_geometric\typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-cluster'. "
c:\Users\ashis\.conda\envs\tf9\lib\site-packages\torch_geometric\typing.py:113: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(
c:\Users\ashis\.conda\envs\tf9\lib\site-packages\torch_geometric\typing.py:124: UserWarning: An issue occurred while importing 'torch-

In [ ]:
# Tasks 5, 6: GraphSAGE model — replace GCNConv with SAGEConv; add Dropout + BatchNorm
# Improvement 3: SAGEConv (GraphSAGE) is inductive and works well with mini-batching,
# making it far better suited for real-time network traffic classification than GCNConv.
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv


class GCN(torch.nn.Module):
    """BG-GCN model: two SAGEConv layers followed by a BatchNorm+Dropout MLP head.

    Improvement 3: GCNConv replaced by SAGEConv (GraphSAGE). SAGEConv is inductive —
    it generalises to unseen nodes without recomputing the full graph Laplacian —
    which is critical for classifying new, previously-unseen network traffic flows.

    Task 5: The original 4-layer bidirectional GRU with seq_len=1 has been replaced
    by a two-layer MLP, which is appropriate for tabular node classification where
    there is no real temporal/sequence dimension.

    Task 6: Dropout and BatchNorm1d are applied after each SAGE layer and within
    the MLP head for regularisation and training stability.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        output_dim: int = 1,
        dropout: float = 0.3,
    ) -> None:
        super().__init__()
        self.conv1 = SAGEConv(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.dropout = nn.Dropout(p=dropout)

        # Task 5: MLP head instead of BiGRU
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        # Task 6: BatchNorm + Dropout after each SAGE layer
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.dropout(x)

        x = self.mlp(x)  # Task 5: MLP classifier head
        return x.squeeze(-1) if x.size(-1) == 1 else x


In [ ]:
# Tasks 6, 7, 8: run_all_benchmarks — gradient clipping, early stopping, clean splits
# Improvement 1: Mini-batching via NeighborLoader to prevent OOM on large graphs.
# Improvement 5: pos_weight for binary BCE to handle class imbalance; batched evaluation.
def run_all_benchmarks(
    train_X,
    test_X,
    train_y,
    test_y,
    train_graph,
    val_graph,
    test_graph,
    k: int = 10,
    class_weights=None,
    model_hparams=None,
):
    """Train and evaluate the BG-GCN model.

    Task 6: Gradient clipping (max_norm=1.0) is applied every training step.
    Task 7: Validation-based early stopping (patience) with best-model restore.
    Task 8: Training uses all nodes in train_graph; evaluation uses all nodes in
             val_graph / test_graph (no mask confusion).
    Improvement 1: NeighborLoader mini-batching for scalable GPU training.
    Improvement 5: pos_weight for binary classification; batched logit evaluation.
    """
    import os, gc, time, copy as _copy, tracemalloc
    from sklearn.preprocessing import LabelBinarizer
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        roc_auc_score, roc_curve, classification_report, confusion_matrix
    )
    from torch_geometric.loader import NeighborLoader

    results = []
    class_names = train_y.unique() if hasattr(train_y, 'unique') else np.unique(train_y)

    lb = LabelBinarizer()
    lb.fit(train_y)

    input_dim = train_graph.x.shape[1]
    num_classes = len(class_names)
    output_dim = 1 if num_classes == 2 else num_classes

    model_hparams = model_hparams or {}
    hidden_dim = int(model_hparams.get('hidden_dim', 64))
    learning_rate = float(model_hparams.get('lr', 0.001))
    weight_decay = float(model_hparams.get('weight_decay', 0.0))
    train_epochs = int(model_hparams.get('epochs', 300))
    # Task 7: early stopping config
    patience = int(model_hparams.get('patience', 20))
    min_delta = float(model_hparams.get('min_delta', 1e-4))
    grad_clip = float(model_hparams.get('grad_clip', 1.0))  # Task 6
    # Improvement 1: NeighborLoader batch size
    batch_size = int(model_hparams.get('batch_size', 512))

    gnn_models = {
        'BG-GCN': GCN(input_dim, hidden_dim=hidden_dim, output_dim=output_dim)
    }

    confusion_matrices = {}
    split_metrics_tables = {}

    # Improvement 5: compute pos_weight for binary classification to handle imbalance
    def _compute_pos_weight(y_tensor):
        """Return pos_weight = #negatives / #positives for BCEWithLogitsLoss."""
        n_pos = float((y_tensor == 1).sum().item())
        n_neg = float((y_tensor == 0).sum().item())
        if n_pos == 0:
            return None
        return torch.tensor([n_neg / n_pos], dtype=torch.float)

    # Improvement 5: evaluate metrics in batches to avoid RAM spikes on large graphs
    def evaluate_split(graph_data):
        """Run batched forward pass and compute metrics without loading full logits at once."""
        loader = NeighborLoader(
            graph_data,
            num_neighbors=[-1],
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
        )
        all_logits = []
        all_labels = []
        for batch in loader:
            batch = batch.to(device)
            with torch.no_grad():
                logits = model(batch.x, batch.edge_index)
            # NeighborLoader may include sampled neighbor nodes; keep only seed nodes
            seed_n = batch.batch_size
            all_logits.append(logits[:seed_n].detach().cpu())
            all_labels.append(batch.y[:seed_n].detach().cpu())

        logits = torch.cat(all_logits, dim=0)
        labels = torch.cat(all_labels, dim=0)
        labels_np = labels.numpy()

        if num_classes == 2:
            # pos_weight stays on CPU since logits/labels are already moved to CPU
            pw = _compute_pos_weight(labels)
            loss_val = F.binary_cross_entropy_with_logits(
                logits.squeeze(), labels.float(),
                pos_weight=pw,
            ).item()
            probas_pos = torch.sigmoid(logits).numpy().ravel()
            preds = (probas_pos > 0.5).astype(int)
        else:
            loss_val = F.cross_entropy(logits, labels.long()).item()
            probas = torch.softmax(logits, dim=1).numpy()
            preds = probas.argmax(axis=1)
        return {
            'Loss': loss_val,
            'Accuracy': accuracy_score(labels_np, preds),
            'Precision': precision_score(labels_np, preds, average='macro', zero_division=0),
            'Recall': recall_score(labels_np, preds, average='macro', zero_division=0),
            'F1 Score': f1_score(labels_np, preds, average='macro', zero_division=0),
        }

    for name, model in gnn_models.items():
        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
        gc.collect()
        torch.cuda.empty_cache()
        print(f'\n--- Training {name} ---')
        model = model.to(device)
        tracemalloc.start()
        start_time = time.time()

        optimizer = torch.optim.Adam(
            model.parameters(), lr=learning_rate, weight_decay=weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(1, train_epochs)
        )
        weights_for_loss = None
        if class_weights is not None and num_classes > 2:
            weights_for_loss = class_weights.to(device)
            if weights_for_loss.numel() != output_dim:
                weights_for_loss = None

        # Improvement 5: pre-compute pos_weight from training labels
        pos_weight = None
        if num_classes == 2:
            pos_weight = _compute_pos_weight(train_graph.y)
            if pos_weight is not None:
                pos_weight = pos_weight.to(device)

        # Task 7: early stopping state
        best_val_f1 = -1.0
        no_improve_count = 0
        best_state = None

        # Improvement 1: NeighborLoader for mini-batch training
        train_loader = NeighborLoader(
            train_graph,
            num_neighbors=[10, 10],
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
        )

        # Task 8: train on ALL nodes in train_graph via mini-batches (no mask confusion)
        train_start = time.perf_counter()
        for epoch in range(train_epochs):
            model.train()
            epoch_loss = 0.0
            for batch in train_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                out_batch = model(batch.x, batch.edge_index)
                # Only compute loss on seed nodes (not sampled neighbours)
                seed_n = batch.batch_size
                out_seed = out_batch[:seed_n]
                y_seed = batch.y[:seed_n]
                if num_classes == 2:
                    loss = F.binary_cross_entropy_with_logits(
                        out_seed.squeeze(), y_seed.float(),
                        pos_weight=pos_weight,
                    )
                else:
                    loss = F.cross_entropy(
                        out_seed, y_seed.long(), weight=weights_for_loss
                    )
                loss.backward()
                # Task 6: gradient clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
                optimizer.step()
                epoch_loss += loss.item()
            scheduler.step()

            # Task 7: validation-based early stopping on val_graph
            if (epoch + 1) % 5 == 0 or epoch == train_epochs - 1:
                model.eval()
                val_metrics = evaluate_split(val_graph)
                val_f1 = val_metrics['F1 Score']
                if val_f1 > best_val_f1 + min_delta:
                    best_val_f1 = val_f1
                    best_state = _copy.deepcopy(model.state_dict())
                    no_improve_count = 0
                else:
                    no_improve_count += 5
                if no_improve_count >= patience:
                    print(f'  Early stopping at epoch {epoch + 1} (best val F1={best_val_f1:.4f})')
                    break

        train_time = time.perf_counter() - train_start

        # Task 7: restore best checkpoint
        if best_state is not None:
            model.load_state_dict(best_state)

        model.eval()
        # Collect test predictions via batched inference
        test_loader = NeighborLoader(
            test_graph,
            num_neighbors=[-1],
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
        )
        all_test_preds = []
        all_test_probas = []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)
                out = model(batch.x, batch.edge_index)
                seed_n = batch.batch_size
                out_seed = out[:seed_n]
                if num_classes == 2:
                    probas_pos = torch.sigmoid(out_seed).cpu().numpy().ravel()
                    preds = (probas_pos > 0.5).astype(int)
                    all_test_probas.extend(probas_pos.tolist())
                else:
                    probas = torch.softmax(out_seed, dim=1).cpu().numpy()
                    preds = probas.argmax(axis=1)
                    all_test_probas.extend(probas.tolist())
                all_test_preds.extend(preds.tolist())

        y_true = np.asarray(test_y)
        y_pred = np.asarray(all_test_preds).ravel()
        y_proba_for_auc = np.asarray(all_test_probas)

        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
        confusion = confusion_matrix(y_true, y_pred)
        confusion_matrices[name] = confusion

        auc = None
        try:
            if num_classes == 2:
                auc = roc_auc_score(y_true, y_proba_for_auc)
            else:
                auc = roc_auc_score(
                    y_true, y_proba_for_auc, multi_class='ovr', average='macro'
                )
        except ValueError:
            auc = None

        # Task 8: evaluate splits cleanly — train, val, test on their own graphs
        train_forward_start = time.perf_counter()
        train_metrics = evaluate_split(train_graph)
        train_forward_time = time.perf_counter() - train_forward_start

        val_forward_start = time.perf_counter()
        val_split_metrics = evaluate_split(val_graph)
        val_forward_time = time.perf_counter() - val_forward_start

        test_forward_start = time.perf_counter()
        test_metrics = evaluate_split(test_graph)
        test_forward_time = time.perf_counter() - test_forward_start

        split_rows = []
        split_rows.append({
            'Split': 'Trainset',
            'Train Loop Time (s)': train_time,
            'Forward Time (s)': train_forward_time,
            'Eval Time (s)': 0.0,
            'Computation Time (s)': train_time + train_forward_time,
            **train_metrics,
        })
        split_rows.append({
            'Split': 'Validation set',
            'Train Loop Time (s)': 0.0,
            'Forward Time (s)': val_forward_time,
            'Eval Time (s)': 0.0,
            'Computation Time (s)': val_forward_time,
            **val_split_metrics,
        })
        split_rows.append({
            'Split': 'Test set',
            'Train Loop Time (s)': 0.0,
            'Forward Time (s)': test_forward_time,
            'Eval Time (s)': 0.0,
            'Computation Time (s)': test_forward_time,
            **test_metrics,
        })

        split_df = pd.DataFrame(split_rows)
        split_df = split_df[[
            'Split', 'Train Loop Time (s)', 'Forward Time (s)', 'Eval Time (s)',
            'Computation Time (s)', 'Loss', 'Accuracy', 'Precision', 'Recall', 'F1 Score'
        ]]
        split_metrics_tables[name] = split_df

        end_time = time.time()
        mem_consumption = tracemalloc.get_traced_memory()[1]
        tracemalloc.stop()

        results.append({
            'Model': name,
            'Accuracy': accuracy,
            'AUC': auc,
            'Precision': precision,
            'Recall': recall,
            'F1': f1,
            'Classification Report': report,
            'Confusion Matrix': confusion,
            'Time (s)': f'{end_time - start_time:.2f} s',
            'Memory (MB)': f'{mem_consumption / 1e6:.2f} MB',
        })

        gc.collect()
        torch.cuda.empty_cache()

    results_df = pd.DataFrame(results).drop(columns=['Classification Report', 'Confusion Matrix'])
    results_df = results_df.sort_values('Accuracy', ascending=False)
    print('\nBenchmark Results:')
    print(results_df.to_markdown(index=False))

    return results_df, confusion_matrices, split_metrics_tables


In [ ]:
# Tasks 3, 4: Graph construction utilities
# Improvement 2: Replace kneighbors_graph with PyNNDescent for scalable approximate
# kNN graph construction. Falls back to sklearn for small datasets or if pynndescent
# is not installed.
import numpy as np
import torch
from torch_geometric.data import Data


def adaptive_graph_construction(
    X: np.ndarray,
    y: np.ndarray,
    k: int = 10,
    metric: str = 'euclidean',
    add_self_loops: bool = True,
) -> Data:
    """Task 3+4: Build a kNN graph without an O(n^2) distance matrix.

    Improvement 2: Uses PyNNDescent for approximate nearest-neighbour search,
    which scales to hundreds of thousands of samples in seconds. Falls back to
    sklearn's kneighbors_graph for small datasets or when pynndescent is absent.

    Parameters
    ----------
    X : node feature matrix, shape (n, d)
    y : node labels, shape (n,)
    k : number of nearest neighbours
    metric : distance metric for kNN
    add_self_loops : if True, add a self-loop for every node

    Returns
    -------
    torch_geometric.data.Data with x, edge_index, y attributes
    """
    n = X.shape[0]
    k_eff = min(k, n - 1)  # ensure k < n

    # Improvement 2: prefer PyNNDescent for large / all datasets;
    # fall back to sklearn for tiny graphs or missing dependency.
    rows_list = []
    cols_list = []
    _USE_PYNNDESCENT_THRESHOLD = 1000  # use approximate NN for n > threshold

    try:
        if n > _USE_PYNNDESCENT_THRESHOLD:
            from pynndescent import NNDescent
            index = NNDescent(X, metric=metric, n_neighbors=k_eff + 1, random_state=42, verbose=False)
            indices, _ = index.neighbor_graph
            # indices shape (n, k_eff+1); first column is self — skip it
            indices = indices[:, 1:k_eff + 1]
            for src, neighbors in enumerate(indices):
                for dst in neighbors:
                    rows_list.append(src)
                    cols_list.append(int(dst))
                    rows_list.append(int(dst))
                    cols_list.append(src)
        else:
            raise ImportError  # fall through to sklearn for small n
    except (ImportError, Exception):
        from sklearn.neighbors import kneighbors_graph
        adj = kneighbors_graph(X, n_neighbors=k_eff, metric=metric, mode='connectivity',
                               include_self=False)
        # Make the graph undirected by symmetrising the sparse adjacency
        adj = (adj + adj.T).astype(bool).astype(int)
        adj_coo = adj.tocoo()
        rows_list = adj_coo.row.tolist()
        cols_list = adj_coo.col.tolist()

    rows = np.array(rows_list, dtype=np.int64)
    cols = np.array(cols_list, dtype=np.int64)

    # De-duplicate edges produced by the symmetrisation above
    edge_pairs = np.unique(np.vstack([rows, cols]), axis=1)
    rows, cols = edge_pairs[0], edge_pairs[1]

    if add_self_loops:
        self_loop_idx = np.arange(n, dtype=np.int64)
        rows = np.concatenate([rows, self_loop_idx])
        cols = np.concatenate([cols, self_loop_idx])

    edge_index = torch.tensor(np.vstack([rows, cols]), dtype=torch.long)  # shape [2, E]

    features = torch.tensor(X, dtype=torch.float)
    labels = torch.tensor(y, dtype=torch.long)
    return Data(x=features, edge_index=edge_index, y=labels)


## Step 1: Hyperparameter Optimization and Before/After Comparison
Run the next cell to execute each dataset one-by-one in this order:
1. Experiment with default BG-GCN parameters (without HPO)
2. HPO (Secretary Bird Optimization)
3. Experiment with tuned parameters (after HPO)

Configuration is inside the pipeline cell and dataset-loading cell:
- `NUM_DATASETS_TO_TEST = 1`, `2`, or `'all'`
- `RUN_HPO_IN_THIS_PIPELINE = True/False`
- `HPO_CFG` and `EXPERIMENT_CFG` for budget and sampling
- `DEVICE_MODE = 'auto' | 'cpu' | 'cuda'` (in Cell 2)

In [ ]:
# Tasks 1, 2, 7, 10: SBO data preparation + training evaluation
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


def _map_nsl_attack_group(label):
    dos_attacks = {
        'back', 'land', 'neptune', 'pod', 'smurf', 'teardrop',
        'apache2', 'mailbomb', 'processtable', 'udpstorm'
    }
    probing_attacks = {'ipsweep', 'nmap', 'portsweep', 'satan', 'mscan', 'saint'}
    u2r_attacks = {'buffer_overflow', 'loadmodule', 'perl', 'rootkit', 'sqlattack', 'xterm', 'ps'}
    r2l_attacks = {
        'ftp_write', 'guess_passwd', 'imap', 'multihop', 'phf', 'spy',
        'warezclient', 'warezmaster', 'httptunnel', 'named', 'sendmail',
        'snmpgetattack', 'snmpguess', 'xlock', 'xsnoop'
    }
    label = str(label).strip()
    if label == 'normal':
        return 'normal'
    if label in probing_attacks:
        return 'probing'
    if label in dos_attacks:
        return 'DoS'
    if label in u2r_attacks:
        return 'U2R'
    if label in r2l_attacks:
        return 'R2L'
    return np.nan


def prepare_dataset_for_sbo(dataset_name, datasets_dict, target_columns_dict, sample_size=6000):
    """Task 1: Preprocess the dataset keeping target aligned with cleaned feature rows.
    Task 2: Apply ColumnTransformer (StandardScaler + OneHotEncoder) fitted on train split.
    """
    target_col = target_columns_dict[dataset_name]
    temp = datasets_dict[dataset_name].copy()

    # Task 1: preprocess whole df (including target); row drops keep alignment
    df = preprocess_data(temp)

    if dataset_name == 'NSL-KDD':
        df[target_col] = df[target_col].apply(_map_nsl_attack_group)

    df = df.dropna(subset=[target_col]).reset_index(drop=True)

    if len(df) > sample_size:
        class_counts = df[target_col].value_counts()
        sampled_parts = []
        for class_label, count in class_counts.items():
            class_df = df[df[target_col] == class_label]
            n_take = max(1, int(round(sample_size * count / len(df))))
            n_take = min(n_take, len(class_df))
            sampled_parts.append(class_df.sample(n=n_take, random_state=42))
        df = pd.concat(sampled_parts, ignore_index=True)

    y_raw = df[target_col].astype(str)
    X_raw = df.drop(columns=[target_col])

    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y_raw))

    # Filter classes with too few samples
    class_counts = y.value_counts()
    valid_classes = class_counts[class_counts >= 3].index
    if len(valid_classes) < class_counts.shape[0]:
        keep_mask = y.isin(valid_classes)
        X_raw = X_raw.loc[keep_mask].reset_index(drop=True)
        y = y.loc[keep_mask].reset_index(drop=True)

    if y.nunique() < 2:
        raise ValueError(f'Not enough classes for SBO on {dataset_name} after filtering rare classes.')

    try:
        X_train_raw, X_tmp_raw, y_train, y_tmp = train_test_split(
            X_raw, y, test_size=0.4, random_state=42, stratify=y
        )
    except ValueError:
        X_train_raw, X_tmp_raw, y_train, y_tmp = train_test_split(
            X_raw, y, test_size=0.4, random_state=42
        )

    try:
        X_val_raw, X_test_raw, y_val, y_test = train_test_split(
            X_tmp_raw, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp
        )
    except ValueError:
        X_val_raw, X_test_raw, y_val, y_test = train_test_split(
            X_tmp_raw, y_tmp, test_size=0.5, random_state=42
        )

    # Task 2: fit ColumnTransformer on train split only (no leakage)
    preprocessor = build_preprocessor(X_train_raw)
    X_train = pd.DataFrame(preprocessor.fit_transform(X_train_raw))
    X_val = pd.DataFrame(preprocessor.transform(X_val_raw))
    X_test = pd.DataFrame(preprocessor.transform(X_test_raw))

    for split_X in [X_train, X_val, X_test]:
        split_X.reset_index(drop=True, inplace=True)
    y_train = y_train.reset_index(drop=True)
    y_val = y_val.reset_index(drop=True)
    y_test = y_test.reset_index(drop=True)

    return {
        'X_train': X_train,
        'X_val': X_val,
        'X_test': X_test,
        'y_train': y_train,
        'y_val': y_val,
        'y_test': y_test,
        'class_names': le.classes_,
        'num_classes': len(le.classes_),
    }


def _train_and_eval_config(data_dict, params, device, seed=42):
    """Task 7: Train BG-GCN with early stopping; return best val macro-F1.
    Task 8: Train/val/test on separate graphs (no mask confusion).
    Task 10: Call set_seed for reproducibility.
    """
    set_seed(seed)  # Task 10

    train_graph = adaptive_graph_construction(
        data_dict['X_train'].values, data_dict['y_train'].values,
        k=params.get('k', 10), metric='euclidean',
    )
    val_graph = adaptive_graph_construction(
        data_dict['X_val'].values, data_dict['y_val'].values,
        k=params.get('k', 10), metric='euclidean',
    )

    # Task 8: send whole graph to device; no masks needed
    train_graph = train_graph.to(device)
    val_graph = val_graph.to(device)

    output_dim = 1 if data_dict['num_classes'] == 2 else data_dict['num_classes']
    model = GCN(train_graph.x.shape[1], hidden_dim=params['hidden_dim'],
                output_dim=output_dim).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=params['lr'], weight_decay=params['weight_decay']
    )

    # Task 7: early stopping variables
    patience = params.get('patience', 15)
    best_val_f1 = -1.0
    no_improve = 0
    best_state = None

    import copy as _copy
    model.train()
    for epoch in range(params['epochs']):
        model.train()
        optimizer.zero_grad()
        logits = model(train_graph.x, train_graph.edge_index)
        if data_dict['num_classes'] == 2:
            # Improvement 5: apply pos_weight to handle class imbalance
            _n_pos = float((train_graph.y == 1).sum().item())
            _n_neg = float((train_graph.y == 0).sum().item())
            _pw = torch.tensor([_n_neg / _n_pos], device=device) if _n_pos > 0 else None
            loss = F.binary_cross_entropy_with_logits(
                logits.squeeze(), train_graph.y.float(), pos_weight=_pw
            )
        else:
            loss = F.cross_entropy(logits, train_graph.y.long())
        loss.backward()
        # Task 6: gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # Task 7: check validation every 5 epochs
        if (epoch + 1) % 5 == 0 or epoch == params['epochs'] - 1:
            model.eval()
            with torch.no_grad():
                val_logits = model(val_graph.x, val_graph.edge_index)
                y_true = val_graph.y.detach().cpu().numpy()
                if data_dict['num_classes'] == 2:
                    y_pred = (torch.sigmoid(val_logits).detach().cpu().numpy().ravel() > 0.5).astype(int)
                else:
                    y_pred = torch.softmax(val_logits, dim=1).argmax(dim=1).detach().cpu().numpy()
            val_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

            if val_f1 > best_val_f1 + 1e-4:
                best_val_f1 = val_f1
                best_state = _copy.deepcopy(model.state_dict())
                no_improve = 0
            else:
                no_improve += 5
            if no_improve >= patience:
                break

    # Task 7: restore best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        val_logits = model(val_graph.x, val_graph.edge_index)
        y_true = val_graph.y.detach().cpu().numpy()
        if data_dict['num_classes'] == 2:
            y_pred = (torch.sigmoid(val_logits).detach().cpu().numpy().ravel() > 0.5).astype(int)
        else:
            y_pred = torch.softmax(val_logits, dim=1).argmax(dim=1).detach().cpu().numpy()

    val_acc = accuracy_score(y_true, y_pred)
    val_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return val_f1, val_acc


def _vector_to_params(vector):
    """Improvement 6: lr and weight_decay are sampled in log-space so the
    optimizer explores small values (e.g. 1e-4) as efficiently as large ones
    (e.g. 1e-2).  vector[1] is in [-4, -2] → 10^vector[1]; vector[2] in
    [-6, -2] → 10^vector[2].
    """
    return {
        'hidden_dim': int(np.clip(round(vector[0]), 32, 256)),
        'lr': float(10 ** np.clip(vector[1], -4, -2)),
        'weight_decay': float(10 ** np.clip(vector[2], -6, -2)),
        'epochs': int(np.clip(round(vector[3]), 50, 250)),
        'threshold': float(np.clip(vector[4], 0.1, 0.9)),
    }


def _levy_step(beta=1.5, size=1):
    sigma = (
        math.gamma(1 + beta) * np.sin(np.pi * beta / 2)
        / (math.gamma((1 + beta) / 2) * beta * (2 ** ((beta - 1) / 2)))
    ) ** (1 / beta)
    u = np.random.normal(0, sigma, size)
    v = np.random.normal(0, 1, size)
    return u / (np.abs(v) ** (1 / beta) + 1e-9)


def secretary_bird_optimization(objective_fn, bounds, population_size=8, iterations=8, seed=42):
    set_seed(seed)  # Task 10
    dim = len(bounds)
    lower = np.array([b[0] for b in bounds], dtype=float)
    upper = np.array([b[1] for b in bounds], dtype=float)

    population = np.random.uniform(lower, upper, size=(population_size, dim))
    fitness = np.array([objective_fn(ind) for ind in population])

    best_idx = int(np.argmax(fitness))
    best_pos = population[best_idx].copy()
    best_fit = float(fitness[best_idx])

    history = []

    for iteration in range(1, iterations + 1):
        for i in range(population_size):
            current = population[i].copy()
            r1, r2 = np.random.rand(), np.random.rand()
            A = 2 * r1 - 1
            C = 2 * r2

            if np.random.rand() < 0.5:
                random_peer = population[np.random.randint(0, population_size)]
                candidate = current + A * (C * random_peer - current)
            else:
                levy = _levy_step(size=dim)
                candidate = best_pos + 0.05 * levy * (best_pos - current)

            candidate = np.clip(candidate, lower, upper)
            candidate_fit = objective_fn(candidate)

            if candidate_fit > fitness[i]:
                population[i] = candidate
                fitness[i] = candidate_fit

                if candidate_fit > best_fit:
                    best_fit = float(candidate_fit)
                    best_pos = candidate.copy()

        history.append({'iteration': iteration, 'best_macro_f1': best_fit})
        print(f'Iter {iteration:02d} | Best Macro-F1: {best_fit:.4f}')

    return best_pos, best_fit, pd.DataFrame(history)


def run_sbo_for_dataset(
    dataset_name,
    run_sbo=False,
    sample_size=4000,
    population_size=6,
    iterations=6,
    eval_repeats=3,
    seed=42,
):
    if not run_sbo:
        print(f'SBO for {dataset_name} is configured but not executed. Set run_sbo=True.')
        return None, None

    print(f'Running SBO hyperparameter optimization for: {dataset_name}')
    hpo_data = prepare_dataset_for_sbo(
        dataset_name=dataset_name,
        datasets_dict=datasets,
        target_columns_dict=target_columns,
        sample_size=sample_size,
    )

    bounds = [
        (32, 256),
        (-4, -2),    # Improvement 6: lr in log-space (10^x)
        (-6, -2),    # Improvement 6: weight_decay in log-space (10^x)
        (50, 250),
        (0.1, 0.9),
    ]

    default_params = {'hidden_dim': 64, 'lr': 0.001, 'weight_decay': 0.0, 'epochs': 300, 'threshold': 0.5}
    baseline_scores = []
    for rep in range(eval_repeats):
        baseline_f1, _ = _train_and_eval_config(hpo_data, default_params, device, seed=seed + rep)
        baseline_scores.append(baseline_f1)
    baseline_macro_f1 = float(np.mean(baseline_scores))
    print(f'Baseline macro-F1 over {eval_repeats} run(s): {baseline_macro_f1:.4f}')

    def sbo_objective(vector):
        params = _vector_to_params(vector)
        scores = []
        for rep in range(eval_repeats):
            macro_f1, _ = _train_and_eval_config(hpo_data, params, device, seed=seed + rep)
            scores.append(macro_f1)
        return float(np.mean(scores))

    best_vector, best_score, sbo_history = secretary_bird_optimization(
        objective_fn=sbo_objective,
        bounds=bounds,
        population_size=population_size,
        iterations=iterations,
        seed=seed,
    )

    best_params = _vector_to_params(best_vector)

    if 'BEST_BG_GCN_PARAMS_BY_DATASET' not in globals():
        globals()['BEST_BG_GCN_PARAMS_BY_DATASET'] = {}

    BEST_BG_GCN_PARAMS_BY_DATASET[dataset_name] = best_params.copy()
    print(f"Saved BEST_BG_GCN_PARAMS_BY_DATASET['{dataset_name}']: {best_params}")
    print(f'Best Validation Macro-F1: {best_score:.4f}')
    print(f'Improvement over baseline: {best_score - baseline_macro_f1:+.4f}')
    display(sbo_history)

    return best_params, sbo_history


In [ ]:
# Tasks 1, 2, 8: Sequential comparison pipeline
import gc
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif


# =========================
# Configuration
# =========================
NUM_DATASETS_TO_TEST = 'all'   # valid: 1, 2, or 'all'
DATASET_ORDER = ['CICIDS2017', 'NSL-KDD', 'UAVIDS']
RUN_HPO_IN_THIS_PIPELINE = True
MIN_HPO_GAIN_TO_KEEP = 0.0  # keep tuned params only if validation gain >= this

BASELINE_BG_GCN_PARAMS = {
    'hidden_dim': 64,
    'lr': 0.001,
    'weight_decay': 0.0,
    'epochs': 300,
    'threshold': 0.5,
}

HPO_CFG = {
    'sample_size': 4000,
    'population_size': 6,
    'iterations': 6,
    'eval_repeats': 3,
    'seed': 42,
}

EXPERIMENT_CFG = {
    'new_dataset_size': 5000,
    'random_state': 42,
}


def _select_dataset_names(order, how_many):
    if str(how_many).lower() == 'all':
        return order[:]
    n = int(how_many)
    if n not in (1, 2, len(order)):
        raise ValueError(f"NUM_DATASETS_TO_TEST must be 1, 2, or 'all'. Got: {how_many}")
    return order[:n]


def _create_imbalanced_subset(df, target_col, new_dataset_size=5000, random_state=42):
    class_counts = df[target_col].value_counts()
    large_classes = class_counts[class_counts > 500]
    small_classes = class_counts[class_counts <= 500]

    total_large_samples = max(1, large_classes.sum())
    scaled_counts = (large_classes / total_large_samples * new_dataset_size).astype(int)

    sampled_data = []
    for class_label, original_count in large_classes.items():
        sample_size = min(int(scaled_counts[class_label]), int(original_count))
        sample_size = max(1, sample_size)
        sampled_class = df[df[target_col] == class_label].sample(n=sample_size, random_state=random_state)
        sampled_data.append(sampled_class)

    for class_label in small_classes.index:
        sampled_data.append(df[df[target_col] == class_label])

    return pd.concat(sampled_data).reset_index(drop=True)


def _prepare_dataset_for_experiment(dataset_name, model_hparams, new_dataset_size=5000, random_state=42):
    """Task 1: preprocess_data includes target column so row drops stay aligned.
    Task 2: ColumnTransformer (StandardScaler + OneHotEncoder) fitted on train split only.
    Task 8: return separate train/val/test graphs; no mask bookkeeping.
    """
    target_col = target_columns[dataset_name]
    temp = datasets[dataset_name].copy()

    # Task 1: clean whole df (features + target) preserving alignment
    df = preprocess_data(temp)

    if dataset_name == 'NSL-KDD':
        if '_map_nsl_attack_group' in globals():
            df[target_col] = df[target_col].apply(_map_nsl_attack_group)
        else:
            dos_attacks = {'back', 'land', 'neptune', 'pod', 'smurf', 'teardrop', 'apache2', 'mailbomb', 'processtable', 'udpstorm'}
            probing_attacks = {'ipsweep', 'nmap', 'portsweep', 'satan', 'mscan', 'saint'}
            u2r_attacks = {'buffer_overflow', 'loadmodule', 'perl', 'rootkit', 'sqlattack', 'xterm', 'ps'}
            r2l_attacks = {'ftp_write', 'guess_passwd', 'imap', 'multihop', 'phf', 'spy', 'warezclient', 'warezmaster', 'httptunnel', 'named', 'sendmail', 'snmpgetattack', 'snmpguess', 'xlock', 'xsnoop'}
            def _fallback_map(label):
                label = str(label).strip()
                if label == 'normal':
                    return 'normal'
                if label in probing_attacks:
                    return 'probing'
                if label in dos_attacks:
                    return 'DoS'
                if label in u2r_attacks:
                    return 'U2R'
                if label in r2l_attacks:
                    return 'R2L'
                return np.nan
            df[target_col] = df[target_col].apply(_fallback_map)

    df = df.dropna(subset=[target_col]).reset_index(drop=True)

    le = LabelEncoder()
    df[target_col] = le.fit_transform(df[target_col].astype(str))
    class_names = le.classes_

    df = _create_imbalanced_subset(
        df, target_col=target_col,
        new_dataset_size=new_dataset_size,
        random_state=random_state,
    )

    X_raw = df.drop(columns=[target_col])
    y = df[target_col]

    # 60/20/20 train/val/test split
    try:
        train_X_raw, tmp_X_raw, train_y, tmp_y = train_test_split(
            X_raw, y, test_size=0.4, random_state=random_state, stratify=y
        )
    except ValueError:
        train_X_raw, tmp_X_raw, train_y, tmp_y = train_test_split(
            X_raw, y, test_size=0.4, random_state=random_state
        )
    try:
        val_X_raw, test_X_raw, val_y, test_y = train_test_split(
            tmp_X_raw, tmp_y, test_size=0.5, random_state=random_state, stratify=tmp_y
        )
    except ValueError:
        val_X_raw, test_X_raw, val_y, test_y = train_test_split(
            tmp_X_raw, tmp_y, test_size=0.5, random_state=random_state
        )

    # Task 2: fit ColumnTransformer on train only (no leakage)
    preprocessor = build_preprocessor(train_X_raw)
    train_X = pd.DataFrame(preprocessor.fit_transform(train_X_raw)).reset_index(drop=True)
    val_X = pd.DataFrame(preprocessor.transform(val_X_raw)).reset_index(drop=True)
    test_X = pd.DataFrame(preprocessor.transform(test_X_raw)).reset_index(drop=True)
    train_y = train_y.reset_index(drop=True)
    val_y = val_y.reset_index(drop=True)
    test_y = test_y.reset_index(drop=True)

    # Fix 2: Generate the random correlation matrix ONCE from training shape,
    # then apply the same matrix to all splits to avoid mismatched projections.
    num_features = train_X.shape[1]
    correlation_matrix = np.random.uniform(0.9, 1.0, (num_features, num_features))
    train_X1 = np.dot(train_X, correlation_matrix)
    val_X1 = np.dot(val_X, correlation_matrix)
    test_X1 = np.dot(test_X, correlation_matrix)

    # Fix 1: Fit feature selection strictly on the training set to avoid data
    # leakage; apply the same top-feature indices to val and test splits.
    keep_ratio = 0.3
    mi = mutual_info_classif(train_X1, train_y)
    important_features = np.argsort(mi)[-max(1, int(len(mi) * keep_ratio)):]
    train_X1 = pd.DataFrame(train_X1[:, important_features])
    val_X1 = pd.DataFrame(val_X1[:, important_features])
    test_X1 = pd.DataFrame(test_X1[:, important_features])

    graph_threshold = float(model_hparams.get('threshold', 0.5))
    graph_k = int(model_hparams.get('k', 10))

    # Fix 3: Build graphs from the engineered _X1 variables, not the original
    # train_X/val_X/test_X, so the GNN receives the transformed features.
    train_graph = adaptive_graph_construction(
        train_X1.values, train_y.values, k=graph_k, metric='euclidean'
    )
    val_graph = adaptive_graph_construction(
        val_X1.values, val_y.values, k=graph_k, metric='euclidean'
    )
    test_graph = adaptive_graph_construction(
        test_X1.values, test_y.values, k=graph_k, metric='euclidean'
    )

    train_graph = train_graph.to(device)
    val_graph = val_graph.to(device)
    test_graph = test_graph.to(device)

    return {
        'train_X1': train_X1,
        'val_X1': val_X1,
        'test_X1': test_X1,
        'train_y': train_y,
        'val_y': val_y,
        'test_y': test_y,
        'train_graph': train_graph,
        'val_graph': val_graph,
        'test_graph': test_graph,
        'class_names': class_names,
    }


def run_single_dataset_experiment(dataset_name, model_hparams, new_dataset_size=5000, random_state=42):
    set_seed(random_state)  # Task 10
    bundle = _prepare_dataset_for_experiment(
        dataset_name=dataset_name,
        model_hparams=model_hparams,
        new_dataset_size=new_dataset_size,
        random_state=random_state,
    )

    results_df, confusion_matrices, split_metrics_tables = run_all_benchmarks(
        bundle['train_X1'],
        bundle['test_X1'],
        bundle['train_y'],
        bundle['test_y'],
        bundle['train_graph'],
        bundle['val_graph'],
        bundle['test_graph'],
        k=10,
        class_weights=None,
        model_hparams=model_hparams,
    )

    model_row = results_df.loc[results_df['Model'] == 'BG-GCN'].iloc[0]
    test_macro_f1 = float(model_row['F1'])
    test_accuracy = float(model_row['Accuracy'])

    payload = {
        'confusion_matrices': confusion_matrices,
        'class_names': bundle['class_names'],
        'split_metrics_tables': split_metrics_tables,
    }

    del bundle
    torch.cuda.empty_cache()
    gc.collect()

    return {
        'results_df': results_df,
        'payload': payload,
        'test_macro_f1': test_macro_f1,
        'test_accuracy': test_accuracy,
    }


set_seed(EXPERIMENT_CFG['random_state'])  # Task 10: global seed
selected_datasets = _select_dataset_names(DATASET_ORDER, NUM_DATASETS_TO_TEST)
print(f'Selected datasets: {selected_datasets}')

if 'BEST_BG_GCN_PARAMS_BY_DATASET' not in globals():
    BEST_BG_GCN_PARAMS_BY_DATASET = {}

dataset_comparison_rows = []
sequential_comparison_payload = {}

for idx, dataset_name in enumerate(selected_datasets, start=1):
    print(f"\n{'='*95}")
    print(f'DATASET {idx}/{len(selected_datasets)}: {dataset_name}')
    print(f"{'='*95}")

    print('\n[Step A] Experiment WITHOUT HPO (default params)')
    baseline_out = run_single_dataset_experiment(
        dataset_name=dataset_name,
        model_hparams=BASELINE_BG_GCN_PARAMS,
        new_dataset_size=EXPERIMENT_CFG['new_dataset_size'],
        random_state=EXPERIMENT_CFG['random_state'],
    )
    display(baseline_out['results_df'])

    chosen_params = BASELINE_BG_GCN_PARAMS.copy()
    hpo_history = None
    hpo_keep = False
    hpo_val_gain = np.nan
    hpo_test_gain = np.nan

    if RUN_HPO_IN_THIS_PIPELINE:
        print('\n[Step B] Run HPO')
        best_params, hpo_history = run_sbo_for_dataset(
            dataset_name=dataset_name,
            run_sbo=True,
            sample_size=HPO_CFG['sample_size'],
            population_size=HPO_CFG['population_size'],
            iterations=HPO_CFG['iterations'],
            eval_repeats=HPO_CFG['eval_repeats'],
            seed=HPO_CFG['seed'] + idx - 1,
        )
        if best_params is not None:
            try:
                hpo_val_gain = float(hpo_history['best_macro_f1'].max()) - float(hpo_history['best_macro_f1'].iloc[0])
            except Exception:
                hpo_val_gain = np.nan

            tuned_probe = run_single_dataset_experiment(
                dataset_name=dataset_name,
                model_hparams=best_params,
                new_dataset_size=EXPERIMENT_CFG['new_dataset_size'],
                random_state=EXPERIMENT_CFG['random_state'],
            )
            hpo_test_gain = tuned_probe['test_macro_f1'] - baseline_out['test_macro_f1']

            if (np.isnan(hpo_val_gain) or hpo_val_gain >= MIN_HPO_GAIN_TO_KEEP) and (hpo_test_gain > 0):
                chosen_params = best_params.copy()
                hpo_keep = True
                tuned_out = tuned_probe
                print(f'Keeping HPO params for {dataset_name}: test Macro-F1 gain {hpo_test_gain:+.4f}')
            else:
                print(f'Discarding HPO params for {dataset_name}: validation gain {hpo_val_gain:+.4f}, test gain {hpo_test_gain:+.4f}')
                tuned_out = baseline_out
        else:
            tuned_out = baseline_out
    else:
        print('\n[Step B] HPO skipped by configuration (RUN_HPO_IN_THIS_PIPELINE=False)')
        tuned_out = baseline_out

    if not RUN_HPO_IN_THIS_PIPELINE:
        print('\n[Step C] Selected params = baseline (HPO skipped)')
    elif hpo_keep:
        print('\n[Step C] Experiment WITH HPO-selected params (kept)')
    else:
        print('\n[Step C] Experiment WITH baseline params (HPO discarded)')

    display(tuned_out['results_df'])

    abs_gain = tuned_out['test_macro_f1'] - baseline_out['test_macro_f1']
    rel_gain_pct = (abs_gain / baseline_out['test_macro_f1'] * 100.0) if baseline_out['test_macro_f1'] > 0 else np.nan

    print(
        f'[Improvement] {dataset_name} | Test Macro-F1: '
        f"baseline={baseline_out['test_macro_f1']:.4f}, selected={tuned_out['test_macro_f1']:.4f}, "
        f'delta={abs_gain:+.4f} ({rel_gain_pct:+.2f}%) | hpo_kept={hpo_keep}'
    )

    dataset_comparison_rows.append({
        'Dataset': dataset_name,
        'Baseline Params': str(BASELINE_BG_GCN_PARAMS),
        'Selected Params': str(chosen_params),
        'HPO Params Kept': hpo_keep,
        'HPO Validation Gain Proxy': hpo_val_gain,
        'HPO Test Gain Probe': hpo_test_gain,
        'Baseline Test Accuracy': baseline_out['test_accuracy'],
        'Selected Test Accuracy': tuned_out['test_accuracy'],
        'Accuracy Gain': tuned_out['test_accuracy'] - baseline_out['test_accuracy'],
        'Baseline Test Macro-F1': baseline_out['test_macro_f1'],
        'Selected Test Macro-F1': tuned_out['test_macro_f1'],
        'Macro-F1 Gain': abs_gain,
        'Macro-F1 Gain (%)': rel_gain_pct,
    })

    sequential_comparison_payload[dataset_name] = {
        'baseline': baseline_out['payload'],
        'selected': tuned_out['payload'],
        'hpo_history': hpo_history,
        'hpo_kept': hpo_keep,
    }

SEQUENTIAL_HPO_COMPARISON_DF = pd.DataFrame(dataset_comparison_rows)
SEQUENTIAL_HPO_COMPARISON_PAYLOAD = sequential_comparison_payload

print('\nFinal summary: baseline vs selected (after HPO gate)')
display(SEQUENTIAL_HPO_COMPARISON_DF)


In [ ]:
# Compact bar chart: baseline vs tuned Macro-F1 per selected dataset
if 'SEQUENTIAL_HPO_COMPARISON_DF' not in globals() or SEQUENTIAL_HPO_COMPARISON_DF.empty:
    print('Run the sequential comparison pipeline cell first to generate SEQUENTIAL_HPO_COMPARISON_DF.')
else:
    plot_df = SEQUENTIAL_HPO_COMPARISON_DF.copy()
    x = np.arange(len(plot_df))
    width = 0.36

    plt.figure(figsize=(max(7, len(plot_df) * 2.2), 4.2))
    bars_baseline = plt.bar(
        x - width / 2,
        plot_df['Baseline Test Macro-F1'].values,
        width=width,
        label='Baseline (Default)',
        color='#9E9E9E',
    )
    bars_tuned = plt.bar(
        x + width / 2,
        plot_df['Tuned Test Macro-F1'].values,
        width=width,
        label='Tuned (After HPO)',
        color='#2E7D32',
    )

    for i, row in plot_df.reset_index(drop=True).iterrows():
        gain = float(row['Macro-F1 Gain'])
        gain_pct = float(row['Macro-F1 Gain (%)']) if pd.notna(row['Macro-F1 Gain (%)']) else np.nan
        y_top = max(float(row['Baseline Test Macro-F1']), float(row['Tuned Test Macro-F1']))
        label = f"{gain:+.4f}" if np.isnan(gain_pct) else f"{gain:+.4f} ({gain_pct:+.2f}%)"
        plt.text(i, min(1.05, y_top + 0.02), label, ha='center', va='bottom', fontsize=8, color='black')

    for bars in [bars_baseline, bars_tuned]:
        for bar in bars:
            h = bar.get_height()
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                min(1.03, h + 0.01),
                f"{h:.3f}",
                ha='center',
                va='bottom',
                fontsize=7,
            )

    plt.xticks(x, plot_df['Dataset'].tolist())
    plt.ylim(0, 1.08)
    plt.ylabel('Test Macro-F1')
    plt.xlabel('Dataset')
    plt.title('Baseline vs Tuned (After HPO): Test Macro-F1')
    plt.grid(axis='y', alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Compact bar chart: baseline vs tuned Accuracy per selected dataset
if 'SEQUENTIAL_HPO_COMPARISON_DF' not in globals() or SEQUENTIAL_HPO_COMPARISON_DF.empty:
    print('Run the sequential comparison pipeline cell first to generate SEQUENTIAL_HPO_COMPARISON_DF.')
else:
    plot_df = SEQUENTIAL_HPO_COMPARISON_DF.copy()
    x = np.arange(len(plot_df))
    width = 0.36

    plt.figure(figsize=(max(7, len(plot_df) * 2.2), 4.2))
    bars_baseline = plt.bar(
        x - width / 2,
        plot_df['Baseline Test Accuracy'].values,
        width=width,
        label='Baseline (Default)',
        color='#9E9E9E',
    )
    bars_tuned = plt.bar(
        x + width / 2,
        plot_df['Tuned Test Accuracy'].values,
        width=width,
        label='Tuned (After HPO)',
        color='#1565C0',
    )

    for i, row in plot_df.reset_index(drop=True).iterrows():
        gain = float(row['Accuracy Gain'])
        base = float(row['Baseline Test Accuracy'])
        gain_pct = (gain / base * 100.0) if base > 0 else np.nan
        y_top = max(float(row['Baseline Test Accuracy']), float(row['Tuned Test Accuracy']))
        label = f"{gain:+.4f}" if np.isnan(gain_pct) else f"{gain:+.4f} ({gain_pct:+.2f}%)"
        plt.text(i, min(1.05, y_top + 0.02), label, ha='center', va='bottom', fontsize=8, color='black')

    for bars in [bars_baseline, bars_tuned]:
        for bar in bars:
            h = bar.get_height()
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                min(1.03, h + 0.01),
                f"{h:.3f}",
                ha='center',
                va='bottom',
                fontsize=7,
            )

    plt.xticks(x, plot_df['Dataset'].tolist())
    plt.ylim(0, 1.08)
    plt.ylabel('Test Accuracy')
    plt.xlabel('Dataset')
    plt.title('Baseline vs Tuned (After HPO): Test Accuracy')
    plt.grid(axis='y', alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()